In [17]:
!pip install -q torch torchvision torchaudio transformers accelerate sentence-transformers faiss-cpu

In [18]:
import re
import faiss
import pickle
import torch
import numpy as np

from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [19]:
device = "cuda"

embedder = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device=device
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3936.68it/s]


In [20]:
def load_docs(base_path):
    docs = []

    for file in Path(base_path).rglob("chapter*.txt"):
        text = file.read_text(encoding="utf-8", errors="ignore").strip()

        parts = str(file).split("/")

        try:
            scripture = parts[1]
            canto = [p for p in parts if "canto" in p][0]
            chapter = file.stem.replace("chapter", "")
        except:
            scripture = "unknown"
            canto = "unknown"
            chapter = file.stem

        docs.append({
            "text": text,
            "source": scripture,
            "canto": canto,
            "chapter": chapter,
            "path": str(file)
        })

    return docs

In [21]:
bg_docs = load_docs("dataset/bg")
sb_docs = load_docs("dataset/sb/canto1")

print("BG chapters:", len(bg_docs))
print("SB chapters:", len(sb_docs))

BG chapters: 18
SB chapters: 19


In [22]:
def clean_scripture(text):
    text = text.replace("PURPORT", ". ")
    text = text.replace("Synonyms", ". ")
    text = text.replace("TRANSLATION", ". ")
    return text

In [23]:
def chunk_text(text, chunk_size=800, overlap=100):

    sentences = text.split(". ")

    chunks = []
    current = ""

    for s in sentences:
        if len(current) + len(s) < chunk_size:
            current += s + ". "
        else:
            chunks.append(current.strip())
            current = s + ". "

    if current:
        chunks.append(current.strip())

    return chunks

In [24]:
ENTITY_LIST = [
    "Prahlada",
    "Dhruva",
    "Gajendra",
    "Ambarisha",
    "Hiranyakashipu",
    "Narasimha",
    "Narada",
    "Kapila",
    "Bharata"
]

In [25]:
def extract_entities(text):

    found = []

    for e in ENTITY_LIST:
        if e.lower() in text.lower():
            found.append(e)

    return found

In [26]:
def final_clean(text):
    return text.replace("\n", " ").strip()

In [27]:
bg_verses = []

for file in Path("dataset/bg").rglob("chapter*.txt"):

    text = file.read_text(encoding="utf-8", errors="ignore")
    text = text.replace("\r", "\n")

    chapter_match = re.search(r"chapter(\d+)", str(file).lower())
    chapter = chapter_match.group(1) if chapter_match else "unknown"

    # proper verse split
    matches = re.split(r"(TEXT\s+\d+)", text)

    verse_id = None
    verse_text = ""

    for part in matches:

        part = part.strip()

        if re.match(r"TEXT\s+\d+", part):
            verse_id = part.replace("TEXT", "").strip()
            verse_text = ""
        else:
            verse_text += " " + part

            if verse_id and len(verse_text.strip()) > 50:

                bg_verses.append({
                    "chapter": chapter,
                    "verse_id": verse_id,
                    "text": verse_text.strip(),
                    "entities": extract_entities(verse_text)
                })
                
                print("✅ BG VERSES LOADED:", len(bg_verses))
                print("SAMPLE BG VERSE:", bg_verses[0])

                verse_text = ""

✅ BG VERSES LOADED: 1
SAMPLE BG VERSE: {'chapter': '1', 'verse_id': '1', 'text': "Translation\nDhrtarashtra said: O Sanjaya, after my sons and the sons of Pandu assembled in the place of pilgrimage at Kurukshetra, desiring to fight, what did they do?\n\nPurport\nBhagavad-gita is the widely read theistic science summarized in the Gita-mahatmya (Glorification of the Gita). There it says that one should read Bhagavad-gita very scrutinizingly with the help of a person who is a devotee of Shri Krishna and try to understand it without personally motivated interpretations. The example of clear understanding is there in the Bhagavad-gita itself, in the way the teaching is understood by Arjuna, who heard the Gita directly from the Lord. If someone is fortunate enough to understand the Bhagavad-gita in that line of disciplic succession, without motivated interpretation, then he surpasses all studies of Vedic wisdom, and all scriptures of the world. One will find in the Bhagavad-gita all that is 

In [28]:
chunked_sb_docs = []

for doc in sb_docs:

    text = clean_scripture(doc["text"])

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):

        chunked_sb_docs.append({
            **doc,
            "chunk_id": i,
            "text": chunk,
            "entities": extract_entities(chunk)
        })

print("SB chunks:", len(chunked_sb_docs))

SB chunks: 2166


In [29]:
# --- ALIGNMENT SAFE BG DATA PREPARATION ---
bg_texts = []
bg_index_map = []

for d in bg_verses:
    text = final_clean(d["text"])

    if isinstance(text, str) and len(text.strip()) > 30:
        bg_texts.append(text)
        bg_index_map.append(d)

print("Total BG texts:", len(bg_texts))

# --- batch encoding (faster + stable) ---
bg_embeddings_list = []

batch_size = 128

def batch_iter(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

for i, batch in enumerate(batch_iter(bg_texts, batch_size)):

    emb = embedder.encode(
        batch,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    bg_embeddings_list.append(emb)

    print(f"BG batch {i+1} done")

# --- merge ---
bg_embeddings = np.vstack(bg_embeddings_list)

print("BG embedding shape:", bg_embeddings.shape)

# --- FAISS index ---
bg_index = faiss.IndexFlatIP(bg_embeddings.shape[1])
bg_index.add(bg_embeddings)

print("✅ BG TEXTS COUNT:", len(bg_texts))
print("SAMPLE TEXT:", bg_texts[0][:200])

print("✅ FAISS SIZE:", bg_index.ntotal)

print("BG FAISS index built successfully")

Total BG texts: 627
BG batch 1 done
BG batch 2 done
BG batch 3 done
BG batch 4 done
BG batch 5 done
BG embedding shape: (627, 768)
✅ BG TEXTS COUNT: 627
SAMPLE TEXT: Translation Dhrtarashtra said: O Sanjaya, after my sons and the sons of Pandu assembled in the place of pilgrimage at Kurukshetra, desiring to fight, what did they do?  Purport Bhagavad-gita is the wi
✅ FAISS SIZE: 627
BG FAISS index built successfully


In [30]:
sb_texts = [
    final_clean(d["text"])
    for d in chunked_sb_docs
]

print("Raw SB texts:", len(sb_texts))

Raw SB texts: 2166


In [31]:
15
# --- OPTIMIZED SB EMBEDDING (FAST GPU BATCH) ---

sb_texts = [
    final_clean(d["text"])
    for d in chunked_sb_docs
    if isinstance(d["text"], str) and len(d["text"].strip()) > 80
]

print("Total SB texts:", len(sb_texts))

# Direct batch encoding (NO MANUAL LOOP)
sb_embeddings = embedder.encode(
    sb_texts,
    batch_size=256,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("SB embedding shape:", sb_embeddings.shape)

# FAISS index
sb_index = faiss.IndexFlatIP(sb_embeddings.shape[1])
sb_index.add(sb_embeddings)

print("SB FAISS index built successfully")


Total SB texts: 2166


Batches: 100%|██████████| 9/9 [00:39<00:00,  4.34s/it]

SB embedding shape: (2166, 768)
SB FAISS index built successfully


In [32]:
faiss.write_index(bg_index, "bg.faiss")
faiss.write_index(sb_index, "sb.faiss")

with open("bg_docs.pkl", "wb") as f:
    pickle.dump(bg_verses, f)

with open("sb_docs.pkl", "wb") as f:
    pickle.dump(chunked_sb_docs, f)

In [33]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:02<00:00, 138.95it/s]


In [34]:
def search_bg(question, k=2):

    q_emb = embedder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    
    print("\n🔍 QUERY:", question)

    scores, ids = bg_index.search(q_emb, k * 5)

    results = []

    for score, idx in zip(scores[0], ids[0]):

        doc = bg_index_map[idx]

        text = doc["text"].lower()

        boost = 0

        # -----------------------------
        # DOCTRINE PRIORITY BOOST
        # -----------------------------
        if "surrender" in question.lower():

            if "18.66" in text or "sarva-dharmān parityajya" in text:
                boost += 10

        final_score = float(score) + boost

        results.append({
            "score": final_score,
            "chapter": doc.get("chapter", "unknown"),
            "verse_id": doc.get("verse_id", "unknown"),
            "text": doc["text"]
        })

    results.sort(key=lambda x: x["score"], reverse=True)

    print("Top BG scores:", [round(r["score"], 3) for r in results[:k]])
    print("FAISS IDS:", ids[0])
    print("FAISS SCORES:", scores[0])

    return results[:k]

In [60]:
def search_bg(question, k=2):

    q_emb = embedder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # retrieve more candidates for reranking
    scores, ids = bg_index.search(q_emb, k * 10)

    results = []

    q_lower = question.lower()

    for score, idx in zip(scores[0], ids[0]):

        doc = bg_index_map[idx]

        text = doc["text"]
        text_lower = text.lower()

        final_score = float(score)

        # -----------------------------
        # LIGHT DOCTRINE-AWARE BOOST (NO HARD CODING ANSWERS)
        # -----------------------------

        # surrender-related intent boost
        if any(w in q_lower for w in ["surrender", "take refuge", "whom shall", "ultimate shelter"]):

            if "mām ekaṁ śaraṇaṁ vraja" in text_lower:
                final_score += 2.5

            if "18.66" in str(doc.get("verse_id", "")):
                final_score += 2.0

        # general Bhagavad Gita instruction boost (Bhagavad Gita conclusion chapters)
        if doc.get("chapter") in ["18"]:
            final_score += 0.3

        results.append({
            "score": final_score,
            "chapter": doc.get("chapter", "unknown"),
            "verse_id": doc.get("verse_id", "unknown"),
            "text": doc["text"]
        })

    # sort by adjusted score
    results.sort(key=lambda x: x["score"], reverse=True)

    print("Top BG scores:", [round(r["score"], 3) for r in results[:k]])

    return results[:k]

In [42]:
def classify_query(question):
    q = question.lower()

    if any(w in q for w in ["surrender", "take refuge", "submit", "whom shall", "whom should"]):
        return "philosophy"

    if any(w in q for w in ["story", "devotee", "example", "pastime", "incident"]):
        return "narrative"

    return "general"

In [43]:
def clean(text, max_chars=900):
    return text.replace("\n", " ")[:max_chars]

In [51]:
def format_answer(question, bg_results, sb_results):

    bg = bg_results[0]  # best match
    print("All results:", bg_results)

    bg_block = f"""
📜 BHAGAVAD GITA (PRIMARY)

Chapter: {bg.get('chapter', '?')}
Verse: {bg.get('verse_id', '?')}

Text:
{bg['text'][:300]}...

"""

    sb_block = "\n\n".join([
        f"""
📖 SRIMAD BHAGAVATAM

Canto: {s.get('canto', '?')}
Chapter: {s.get('chapter', '?')}

Text:
{s['text'][:250]}...
"""
        for s in sb_results
    ])

    final = f"""
══════════════════════════════════════
QUESTION:
{question}
══════════════════════════════════════

{bg_block}

{sb_block}

══════════════════════════════════════
🎯 FINAL CONCLUSION

Based on Bhagavad Gita and Srimad Bhagavatam:
Lord Krishna is the ultimate shelter (BG 18.66).
══════════════════════════════════════
"""

    return final

In [45]:
20
def ask(question):

    mode = classify_query(question)

    # -------------------------
    # RETRIEVAL ROUTING
    # -------------------------
    if mode == "philosophy":
        bg_results = search_bg(question, k=5)
        sb_results = search_sb(question, k=2)

    elif mode == "narrative":
        bg_results = search_bg(question, k=2)
        sb_results = search_sb(question, k=5)

    else:
        bg_results = search_bg(question, k=3)
        sb_results = search_sb(question, k=3)

    # -------------------------
    # CONTEXT BUILD (RAW FOR LLM)
    # -------------------------
    bg_context = "\n\n".join(clean(r["text"]) for r in bg_results)
    sb_context = "\n\n".join(clean(r["text"]) for r in sb_results)

    # -------------------------
    # LLM PROMPT (NO FORMATTING RESPONSIBILITY)
    # -------------------------
    prompt = f"""
You are a spiritual knowledge assistant.

Use ONLY the provided context.

Bhagavad Gita is the primary authority.
Srimad Bhagavatam is supporting explanation.

Do NOT format the answer.
Do NOT add headings or structure.

Just explain the meaning clearly and concisely.

QUESTION:
{question}

BG CONTEXT:
{bg_context}

SB CONTEXT:
{sb_context}

ANSWER:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            temperature=0.2
        )

    llm_output = tokenizer.decode(out[0], skip_special_tokens=True)

    # -------------------------
    # FINAL STRUCTURED FORMATTING (PYTHON CONTROLLED)
    # -------------------------
    bg_block = ""
    if bg_results:
        bg = bg_results[0]
        bg_block = f"""
══════════════════════════════════════
📜 BHAGAVAD GITA

Chapter: {bg.get('chapter', '?')}
Verse: {bg.get('verse_id', '?')}

Text:
{clean(bg['text'])[:400]}...
"""

    sb_block = "\n\n".join([
        f"""
📖 SRIMAD BHAGAVATAM

Canto: {s.get('canto', '?')}
Chapter: {s.get('chapter', '?')}

Text:
{clean(s['text'])[:250]}...
"""
        for s in sb_results
    ])

    final_output = f"""
══════════════════════════════════════
QUESTION:
{question}

{bg_block}

══════════════════════════════════════

{sb_block}

══════════════════════════════════════

🎯 EXPLANATION (LLM INSIGHT)

{llm_output}

══════════════════════════════════════
"""

    return final_output

In [61]:
import time

#question = "What are the qualities of a devotee?"
question = "Whom shall we ultimately surrender upon?"

start = time.time()

print(ask(question))

print("\nTime taken:", time.time() - start)

Top BG scores: [0.92, 0.896, 0.887, 0.875, 0.873]

══════════════════════════════════════
QUESTION:
Whom shall we ultimately surrender upon?


══════════════════════════════════════
📜 BHAGAVAD GITA

Chapter: 18
Verse: 62

Text:
Translation O scion of Bharata, surrender unto Him utterly. By His grace you will attain transcendental peace and the supreme and eternal abode.  Purport A living entity should therefore surrender unto the Supreme Personality of Godhead, who is situated in everyone's heart, and that will relieve him from all kinds of miseries of this material existence. By such surrender, not only will one be rele...


══════════════════════════════════════


📖 SRIMAD BHAGAVATAM

Canto: unknown
Chapter: chapter7

Text:
This surrendering process is the remedial measure for getting relief from the bewildering ways of the illusory energy. The surrendering process is completed by the influence of association. The Lord has suggested, therefore, that by the influence of ...



📖 SRIMA